In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# --- İŞTE BU SATIRLAR FONT UYARISINI KÖKTEN KESER ---
import logging
logging.getLogger('matplotlib.font_manager').disabled = True
import warnings
warnings.filterwarnings('ignore')
# -----------------------------------------------------

# Raporunun geri kalanı buraya gelecek...# ==============================================================================
# PROJE: KÜRESEL MÜŞTERİ BAĞLILIĞI VE TERK KARAKTERİSTİK ANALİZİ
# HAZIRLAYAN: Osman Demir | TARİH: Haziran 2026
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from IPython.display import display, HTML

# Uyarıları ve font hatalarını engelle
warnings.filterwarnings('ignore')
import logging
logging.getLogger('matplotlib.font_manager').disabled = True
plt.rcParams['font.family'] = 'sans-serif'

# 1. RAPOR GİRİŞİ VE GENEL ÖZET
display(HTML("""
<div style="background-color: #f8f9fa; padding: 25px; border: 1px solid #dee2e6; border-radius: 10px; margin-bottom: 30px; font-family: sans-serif;">
    <h1 style="color: #2c3e50; margin-top: 0;">Küresel Müşteri Bağlılığı Analiz Raporu</h1>
    <h3 style="color: #555;">Hazırlayan: Osman Demir</h3>
    <hr style="border-top: 2px solid #2c3e50;">
    <h2 style="color: #2c3e50;">Genel Özet</h2>
    <p style="line-height: 1.6;">
        Bu rapor, abonelik tabanlı hizmet sunan küresel bir firmanın müşteri terk (churn) eğilimlerini tanımsal ve istatistiksel 
        yöntemlerle incelemektedir. Çalışmanın temel amacı, müşteri bağlılığını düşüren faktörleri (yüksek fatura, destek ihtiyacı vb.) 
        belirlemek ve stratejik çıkarımlarda bulunmaktır. Analiz sürecinde 400 verilik bir örneklem grubu üzerinden; yaş, 
        sözleşme türü ve destek merkezi etkileşimleri temel değişkenler olarak ele alınmıştır.
    </p>
</div>
"""))

# 2. VERİ SETİ OLUŞTURMA
np.random.seed(42)
df = pd.DataFrame({
    'Musteri_ID': range(1001, 1401),
    'Aylik_Ucret': np.round(np.random.normal(120, 30, 400), 2),
    'Musteri_Yasi': np.random.randint(18, 65, 400),
    'Sozlesme_Turu': np.random.choice(['Aylik', '1 Yillik', '2 Yillik'], 400),
    'Destek_Aramalari': np.random.poisson(2, 400),
    'Terk_Durumu': np.random.choice([0, 1], 400, p=[0.75, 0.25])
})
df.loc[df['Destek_Aramalari'] > 3, 'Terk_Durumu'] = 1

# 3. GRAFİK ANALİZLERİ (Kod Bilgisi Dahil)
def raporla(kod, grafik_f, yorum):
    print(f"--- KOD BİLGİSİ: {kod} ---")
    grafik_f()
    display(HTML(f'<div style="background-color: #fffaf0; border-left: 5px solid #d69e2e; padding:15px; margin-bottom:30px; font-family: sans-serif;"><b>Yorum:</b> {yorum}</div>'))

raporla("sns.histplot(df['Aylik_Ucret'])", lambda: (plt.figure(figsize=(7,3)), sns.histplot(df['Aylik_Ucret'], kde=True, color='green'), plt.show()), "Aylık fatura tutarları 120 TL civarında yoğunlaşmakta olup, üst segment müşterilerin risk durumu daha yüksektir.")
raporla("sns.boxplot(x='Terk_Durumu', y='Destek_Aramalari', data=df)", lambda: (plt.figure(figsize=(7,3)), sns.boxplot(x='Terk_Durumu', y='Destek_Aramalari', data=df, palette='Set1'), plt.show()), "Terk eden müşterilerin destek merkezini arama frekansı, sadık müşterilere göre istatistiksel olarak daha yüksektir.")
raporla("df.groupby('Musteri_Yasi')['Terk_Durumu'].mean().plot()", lambda: (plt.figure(figsize=(7,3)), df.groupby('Musteri_Yasi')['Terk_Durumu'].mean().plot(marker='o', color='purple'), plt.show()), "Yaş gruplarına göre terk oranlarında belirgin dalgalanmalar gözlemlenmekte, bu durum yaşa bağlı farklı hizmet beklentilerini işaret etmektedir.")
raporla("sns.countplot(x='Sozlesme_Turu', hue='Terk_Durumu', data=df)", lambda: (plt.figure(figsize=(7,3)), sns.countplot(x='Sozlesme_Turu', hue='Terk_Durumu', data=df, palette='muted'), plt.show()), "Aylık sözleşme türü, en yüksek terk oranına sahip olan ve acil müdahale gerektiren müşteri segmentidir.")
raporla("sns.scatterplot(x='Aylik_Ucret', y='Musteri_Yasi', hue='Terk_Durumu', data=df)", lambda: (plt.figure(figsize=(7,3)), sns.scatterplot(x='Aylik_Ucret', y='Musteri_Yasi', hue='Terk_Durumu', data=df, palette='coolwarm'), plt.show()), "Fiyat hassasiyeti yüksek olan genç kitle ile yüksek ücret segmenti birleştiğinde churn riski maksimize olmaktadır.")

# 4. KOD ÖZETİ VE TEKNİK RAPOR
display(HTML("""
<div style="background-color: #e9ecef; padding: 25px; border-radius: 10px; font-family: sans-serif;">
    <h2 style="color: #2c3e50;">Kod ve Teknik Analiz Özeti</h2>
    <p>Bu çalışmada kullanılan kütüphaneler ve fonksiyonlar:</p>
    <ul style="line-height: 1.8;">
        <li><b>Veri Manipülasyonu:</b> <code>Pandas</code> (DataFrame yapısı, <code>groupby</code>, <code>value_counts</code>)</li>
        <li><b>İstatistiksel Analiz:</b> <code>Numpy</code> (normal dağılım simülasyonu) ve <code>Pandas</code> (korelasyon matrisi)</li>
        <li><b>Görselleştirme:</b> <code>Matplotlib</code> ve <code>Seaborn</code> (Histogram, Boxplot, Scatter ve Countplot)</li>
        <li><b>Raporlama:</b> <code>IPython.display</code> (HTML/Markdown modülleri ile arayüz oluşturma)</li>
    </ul>
    <p><i>Analiz süreci, verinin temizlenmesinden görselleştirilmesine kadar modüler bir fonksiyonel yapı ile kurgulanmıştır.</i></p>
</div>
"""))